# Results Plots

Box-plots and tables of C-index and IBS across all datasets and methods.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from main import run_dataset, summarize, DATASETS

plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = sns.color_palette('tab10')
os_import = __import__('os'); os_import.makedirs('../figures', exist_ok=True)

In [ ]:
# Run all experiments (or load cached results)
all_results = {ds: run_dataset(ds) for ds in DATASETS}

## Summary Table

In [ ]:
tables = {}
for ds, res in all_results.items():
    tables[ds] = summarize(res)
    print(f"\n{ds}")
    print(tables[ds].to_string())

# Combined LaTeX table
combined = pd.concat(tables, axis=0)
print('\nLaTeX:')
print(combined.to_latex())

## Box-plots: C-index and IBS

In [ ]:
# Flatten results for plotting
records = []
for ds, res in all_results.items():
    for method, metrics in res.items():
        for m in metrics:
            records.append({'dataset': ds, 'method': method,
                            'C-index': m['ci'], 'IBS': m['ibs']})
df_plot = pd.DataFrame(records)

methods = df_plot['method'].unique()
datasets = list(DATASETS.keys())

fig, axes = plt.subplots(2, len(datasets), figsize=(4 * len(datasets), 8), sharey='row')
for col, ds in enumerate(datasets):
    sub = df_plot[df_plot['dataset'] == ds]
    for row, metric in enumerate(['C-index', 'IBS']):
        ax = axes[row, col]
        sns.boxplot(data=sub, x='method', y=metric, ax=ax,
                    palette=PALETTE[:len(methods)], width=0.5, linewidth=1)
        ax.set_title(ds if row == 0 else '', fontsize=11)
        ax.set_xlabel('')
        ax.tick_params(axis='x', rotation=30)
        if col == 0:
            ax.set_ylabel(metric)
        else:
            ax.set_ylabel('')

plt.suptitle('C-index (↑) and IBS (↓) across datasets', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../figures/results_boxplots.pdf', dpi=300, bbox_inches='tight')
plt.savefig('../figures/results_boxplots.png', dpi=300, bbox_inches='tight')
plt.show()